Q1. [50 points]

Data based RL controller implementing PPO algorithm is implemented in rl-wind.ipynb (See
RL/LAB). 

Get familiar with code and see how the different parts – state, observation, rewards,
and algorithm are integrated.


Consider the given algorithm as base line. Modify the reward function or power generated
equation or both. Display all the parameters as shown in the .ipynb and compare your results with
the baseline. The changes in reward function or power equation should be based on physics and
consistent with physics/mathematics.


While presentation explain the states, observation, reward function and relevant parameters in
detail

In [ ]:
# See changes in lab_5/rl-wind-1.ipynb

Q2. Choose ONLY ONE OPTION [50 points]

Option1 :

Leverage cyberboticss webots (as in Assignment 2) to learn a controller that learns to navigate
autonomously with collision avoidance and reaches the predefined goal. Train the model and
save it. 

Then leverage this model to test 5 times – if agent can reach the goal each time. Plot and
compare the trajectories as the agent travels for each test.

While presentation explain the environment, states, observation, reward function and relevant
parameters in detail. Also explain the interface – how webots is connected to stable baseline -
PPO algorithm.

In [ ]:
# Gym setup

import gymnasium as gym
from gymnasium import spaces
import numpy as np

class EPuckChessEnv(gym.Env):
    """
    E-puck navigation environment on an 8x8 chessboard 
    with 3 static obstacles and a goal (yellow duck).
    """
    def __init__(self):
        super().__init__()
        
        self.dt = 0.1  # timestep in seconds
        self.board_size = 8.0  # 8x8 units
        
        # Fixed positions for goal (upper-left) and obstacles
        self.goal_pos = np.array([1.5, 6.5], dtype=np.float32)
        self.obstacles = [
            np.array([3.0, 3.0], dtype=np.float32),
            np.array([5.0, 4.0], dtype=np.float32),
            np.array([2.0, 5.0], dtype=np.float32)
        ]
        
        # Lower right starting position area
        self.start_pos = np.array([7.0, 1.0], dtype=np.float32)
        
        # Observation space: [x, y, dist_to_goal, angle_to_goal, dist_obs1, dist_obs2, dist_obs3]
        low = np.array([0.0, 0.0, 0.0, -np.pi, 0.0, 0.0, 0.0], dtype=np.float32)
        high = np.array([self.board_size, self.board_size, 12.0, np.pi, 10.0, 10.0, 10.0], dtype=np.float32)
        self.observation_space = spaces.Box(low, high, dtype=np.float32)
        
        # Action space: [linear_velocity, angular_velocity]
        self.action_space = spaces.Box(
            low=np.array([-0.2, -1.0], dtype=np.float32),
            high=np.array([0.2, 1.0], dtype=np.float32),
            dtype=np.float32
        )
        
        self.robot_pos = np.copy(self.start_pos)
        self.robot_theta = 0.0
        self.time = 0.0
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.time = 0.0
        self.robot_pos = self.start_pos + np.random.uniform(-0.2, 0.2, size=2)
        self.robot_theta = np.random.uniform(0, 2 * np.pi)
        return self.get_state(), {}

    def get_state(self):
        goal_diff = self.goal_pos - self.robot_pos
        dist_to_goal = np.linalg.norm(goal_diff)
        angle_to_goal = np.arctan2(goal_diff[1], goal_diff[0]) - self.robot_theta
        angle_to_goal = (angle_to_goal + np.pi) % (2 * np.pi) - np.pi
        
        obs_dists = [np.linalg.norm(obs_pos - self.robot_pos) for obs_pos in self.obstacles]
        
        return np.array([
            self.robot_pos[0], self.robot_pos[1],
            dist_to_goal, angle_to_goal,
            obs_dists[0], obs_dists[1], obs_dists[2]
        ], dtype=np.float32)

    def step(self, action):
        self.time += self.dt
        v = float(action[0])
        omega = float(action[1])
        
        # Kinematic update (in Webots, replace this with motor commands and supervisor state reads)
        self.robot_theta += omega * self.dt
        self.robot_pos[0] += v * np.cos(self.robot_theta) * self.dt
        self.robot_pos[1] += v * np.sin(self.robot_theta) * self.dt
        self.robot_pos = np.clip(self.robot_pos, 0.0, self.board_size)
        
        obs = self.get_state()
        dist_to_goal = obs[2]
        
        # Reward shaping: encourage moving towards goal, penalize high angular velocity
        reward = -0.1 * dist_to_goal - 0.02 * abs(omega)
        
        terminated = False
        truncated = False
        
        # Check obstacle collision
        for obs_pos in self.obstacles:
            if np.linalg.norm(obs_pos - self.robot_pos) < 0.35:
                reward -= 10.0
                terminated = True
                
        # Check goal reached (yellow duck)
        if dist_to_goal < 0.25:
            reward += 50.0
            terminated = True
            
        if self.time > 40.0:
            truncated = True
            
        return obs, reward, terminated, truncated, {}

In [ ]:
# PPO training

from stable_baselines3 import PPO

env = EPuckChessEnv()

model = PPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    verbose=1
)

model.learn(total_timesteps=300000)
model.save("ppo_epuck_chess_controller")

In [ ]:
# Supervisor Setup

import os
import sys
import numpy as np

os.environ['WEBOTS_HOME'] = "E:\\Webots"
os.environ['WEBOTS_ROBOT_NAME'] = 'e-puck'

webots_home = os.environ.get('WEBOTS_HOME', '')
if sys.platform == 'win32':
    controller_path = os.path.join(webots_home, 'lib', 'controller', 'python')

if os.path.exists(controller_path):
    sys.path.append(controller_path)
else:
    print(f"Warning: Controller path not found at {controller_path}. Check WEBOTS_HOME.")

try:
    from controller import Supervisor
except ImportError:
    sys.exit("Error: Could not import Webots controller library. Make sure WEBOTS_HOME is configured or run via the Webots python environment.")

class WebotsEPuckSupervisor:
    def __init__(self):
        self.supervisor = Supervisor()
        self.time_step = int(self.supervisor.getBasicTimeStep())
        
        # Get robot and object nodes from Webots
        self.robot_node = self.supervisor.getFromDef("EPUCK")
        self.goal_node = self.supervisor.getFromDef("DUCK")
        self.obstacle_nodes = [
            self.supervisor.getFromDef("BOX_1"),
            self.supervisor.getFromDef("BOX_2"),
            self.supervisor.getFromDef("BOX_3")
        ]
        
        # Motors
        self.left_motor = self.supervisor.getDevice("left wheel motor")
        self.right_motor = self.supervisor.getDevice("right wheel motor")
        self.left_motor.setPosition(float('inf'))
        self.right_motor.setPosition(float('inf'))
        self.left_motor.setVelocity(0.0)
        self.right_motor.setVelocity(0.0)

    def get_observation(self):
        robot_pos = self.robot_node.getPosition()[:2]
        goal_pos = self.goal_node.getPosition()[:2]
        
        obs_dists = [np.linalg.norm(obs.getPosition()[:2] - np.array(robot_pos)) for obs in self.obstacle_nodes]
        goal_diff = np.array(goal_pos) - np.array(robot_pos)
        dist_to_goal = np.linalg.norm(goal_diff)
        
        # Extract orientation (yaw) from rotation matrix if needed
        rotation = self.robot_node.getOrientation()
        # Webots rotation matrix extraction for 2D heading
        robot_theta = np.arctan2(rotation[3], rotation[0]) 
        
        angle_to_goal = np.arctan2(goal_diff[1], goal_diff[0]) - robot_theta
        angle_to_goal = (angle_to_goal + np.pi) % (2 * np.pi) - np.pi
        
        return np.array([
            robot_pos[0], robot_pos[1],
            dist_to_goal, angle_to_goal,
            obs_dists[0], obs_dists[1], obs_dists[2]
        ], dtype=np.float32)

    def apply_action(self, action):
        # Convert linear (v) and angular (omega) velocity to differential wheel speeds
        v, omega = action[0], action[1]
        wheel_radius = 0.0205  # e-puck wheel radius in meters
        axle_track = 0.052     # distance between wheels
        
        left_speed = (v - omega * axle_track / 2.0) / wheel_radius
        right_speed = (v + omega * axle_track / 2.0) / wheel_radius
        
        self.left_motor.setVelocity(np.clip(left_speed, -6.28, 6.28))
        self.right_motor.setVelocity(np.clip(right_speed, -6.28, 6.28))
        
        self.supervisor.step(self.time_step)

--- External e-puck Keyboard Controller Connected ---
Use the arrow keys to drive (focus your terminal or simulation window):
  [Up]    : Move Forward
  [Down]  : Move Backward
  [Left]  : Turn Left
  [Right] : Turn Right
  [Space] : Stop
